# FACE autism spectrum cohort: a first tutorial

This notebook shows how to load the FACE ASP data (autism spectrum disorders, children and adults) with the `facedata` package and how to take a first look at them. It is written for people who have never used this cohort and who do not code much.

You will learn how to:

1. load the data and understand the tables you get;
2. find a variable and read what it means, in a database whose **dictionary arrived after the work**;
3. tell children from adults, and the screening visit from the inclusion visit;
4. judge how much a value can be trusted, and avoid the traps of this cohort;
5. work through four questions: the **diagnostic profile**, **children and adults**, **comorbidities and treatments**, **self-reported traits**;
6. build your own table and save it;
7. **put the bipolar, schizophrenia, depression and autism cohorts in one table.**

**How to use a notebook.** A notebook is made of cells. Click in a grey cell and press `Shift` + `Enter` to run it: the result appears under it. Run the cells from top to bottom, because each one uses what the cells above it made.

## Before you start: these are patient data

The results that appear under the cells are patient data, even if the patients have no name. Many patients of this cohort are children.

- **Before you share this notebook or send it to GitHub, clear the results:** menu *Edit*, then *Clear Outputs of All Cells*, then save.
- Never copy patient rows into an e-mail, a chat or a slide. Share counts, means and figures of groups.
- Save your own tables in the folder `data/exports/`. Everything in `data/` stays on this computer.

## 1. Load the cohort

The first cell brings in the tools. The second one reads the extraction: it takes about a minute (the file has more than four thousand columns), so wait for the number in brackets on the left of the cell to appear.

If your extraction file has another name, change it in the first cell.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from facedata import asp

EXTRACTION = "data/asp/ASPALL_20261030.tsv"        # the file you received
pd.set_option("display.max_columns", 30)

In [ ]:
result = asp.load(EXTRACTION)

`result` holds six tables. You will mostly use the first two.

| Table | One row per | What it holds |
|---|---|---|
| `result.data` | patient visit | everything measured at a visit: interviews, scales, tests, lab results, answers |
| `result.patients` | patient | what does not change: sex, date of birth, the diagnosis, the form filled before the assessment |
| `result.treatments` | patient and item of a list | eleven lists: the treatments of each visit, the diseases of each body system, the relatives, the hospitalisations |
| `result.relatives` | patient and relative | empty in this cohort: the relatives are a list of `treatments` |
| `result.qc` | variable | the catalogue: every variable, its label and how well it is filled |
| `result.flags` | flagged cell | every value that was repaired, doubted or removed, with the reason |

In [ ]:
data = result.data
patients = result.patients

print(len(data), "rows of", data["usubjid"].nunique(), "patients")
print(data.shape[1], "columns in data and", patients.shape[1], "in patients")

## 2. What a visit looks like

Every row of `data` starts with the same columns:

- `cohort`: `asp` on every row. It matters when you put several cohorts together (section 14).
- `usubjid`: the patient's number. It links all the tables.
- `visit`: the name of the visit. `screening` comes first, `V0` is the inclusion visit, then `V1_an` (one year), `V2_ans` and so on.
- `visit_month`: the planned number of months since inclusion (0, 12, 24...). Empty for the screening visit.
- `visit_date`, `siteid` (the centre), `age` (at the visit).

In [ ]:
data["visit"].value_counts().plot.bar(figsize=(9, 3), title="Number of visits of each kind")
plt.show()

Two things stand out, and both shape every analysis of this cohort.

**The screening visit is a visit of its own.** Some questionnaires are filled there and nowhere else (the RAADS-R, the SRS, the AQ). If you look for them on the inclusion row you will find nothing.

**There is almost no follow-up.** Most patients have the inclusion visit only: this cohort is a cross-section, not a follow-up study.

**The pattern to remember:** `table[table["column"] == value]` keeps the rows where the column has that value.

In [ ]:
inclusion = data[data["visit"] == "V0"]
screening = data[data["visit"] == "screening"]

visits_per_patient = data[data["visit"] != "screening"].groupby("usubjid").size()
print("Patients with the inclusion visit only:", (visits_per_patient == 1).sum(), "of", len(visits_per_patient))

inclusion["visit_date"].dt.year.value_counts().sort_index().plot.bar(figsize=(9, 3), title="Inclusion visits per year")
plt.show()

## 3. Children and adults, diagnosed and not diagnosed

`patients` has one row per patient. Two columns matter before anything else:

- `age_at_inclusion`: the cohort holds children from 5 years and adults. Many instruments exist in a child form and an adult form, which are **different columns** (section 7).
- `diagnosis`: `asd_confirmed` when the expert centre confirmed an autism spectrum disorder, `asd_not_confirmed` when the patient was assessed and not diagnosed. Both groups are in the tables: keep the one your question needs.

In [ ]:
print(patients["diagnosis"].value_counts())
print()
print(patients["sex"].value_counts())

patients["age_at_inclusion"].plot.hist(bins=range(0, 70, 2), figsize=(9, 3), title="Age at inclusion")
plt.axvline(18, color="black", linestyle="--")
plt.show()

A column `age_group` will be useful all along. `merge` brings columns of `patients` next to the visits: the two tables share `usubjid`.

In [ ]:
patients["age_group"] = "adult"
patients.loc[patients["age_at_inclusion"] < 18, "age_group"] = "child"

inclusion = inclusion.merge(patients[["usubjid", "sex", "diagnosis", "age_group"]], on="usubjid")
screening = screening.merge(patients[["usubjid", "sex", "diagnosis", "age_group"]], on="usubjid")

pd.crosstab(inclusion["age_group"], inclusion["diagnosis"])

## 4. Find a variable

There are more than four thousand columns. `result.qc` is the catalogue: one row per variable, with its `label`, its `instrument`, the source table of the export (`tab`) and how many visits hold it (`n_valid`).

The function below searches a word in the names and labels. Try `"ADOS"`, `"SRS"`, `"sleep"`, `"school"`.

In [ ]:
qc = result.qc


def find(word):
    """The variables whose name, label or instrument holds the word."""
    found = qc["variable"].str.contains(word, case=False) | qc["label"].str.contains(word, case=False, na=False) | qc["instrument"].str.contains(word, case=False)
    return qc.loc[found, ["variable", "label", "instrument", "tab", "n_valid"]]


find("ADOS").head(15)

The list of the instruments, with the number of variables of each and the largest number of visits that hold one of them:

In [ ]:
qc.groupby(["tab", "instrument"]).agg(variables=("variable", "size"), visits=("n_valid", "max")).sort_values("visits", ascending=False).head(25)

## 5. A dictionary that arrived after the work

The other FACE cohorts came with a dictionary (a "thesaurus"): the wording of each question and each answer. **For this one it arrived late.** Every instrument was first identified from its column names, the number of its items, its answers and its stored scores, compared with the published scale. The thesaurus was then compared with that reading: it confirmed every instrument, corrected eight readings, and is itself wrong in a few places, where the data decide. What this means for you:

- **The notes of an instrument say how it was identified and what the thesaurus changed.** Read them before you use it: the cell below prints them.
- **A list of answers is delivered in words** (`asd_evoked`, `right`, `abuse`...). A code that the thesaurus does not document stays a numbered code, `code_3`, and the label says "not documented".
- **Items are labelled by number** ("ADI-R item 50"), never by their wording: the text of a published questionnaire is not copied.
- **A child form and an adult form are two instruments**, even in the same columns: the SRS filled by a relative is `srsc_...` for a child and `srsr_...` for an adult.

In [ ]:
import yaml
from pathlib import Path


def notes(instrument):
    """Print what is known about an instrument: how it was identified, what was checked, what was corrected."""
    file = next(Path("facedata/asp/schemas").glob(f"*/{instrument}.yaml"))
    schema = yaml.load(file.read_text(encoding="utf-8"), Loader=yaml.BaseLoader)
    print(schema["title"])
    print()
    print(schema["notes"])


notes("srs_relative_child")

How many variables does the thesaurus leave undocumented? The labels tell.

In [ ]:
not_documented = qc["label"].str.contains("not documented", na=False)
print(not_documented.sum(), "variables of", len(qc), "are not documented by the thesaurus")

qc.loc[not_documented, ["instrument", "variable", "label"]].head(12)

## 6. How much can a value be trusted?

Each value was checked when the file was loaded. `result.flags` lists every cell that was repaired, doubted or removed. The column `severity` has three levels:

- `info`: the value was repaired or annotated and can be used;
- `discrepancy`: the value is kept, but something disagrees with it;
- `danger`: the value could not be used and is missing in `data`.

In [ ]:
flags = result.flags

print(flags["severity"].value_counts())
flags["flag"].value_counts()

Three flags are worth knowing in this cohort.

**`score_as_stored`.** The forms let a clinician type a score without its items. Where no item is recorded, the stored score is delivered as it is and flagged. It was not recomputed, because there is nothing to recompute it from.

**`score_mismatch`.** Where the items are there, every score is recomputed from them with the published rule. If the stored score differs, the recomputed one is delivered and the visit is flagged.

**`missing_code`.** A "9" for "not known" is read as missing, with its reason.

In [ ]:
typed_alone = flags[flags["flag"] == "score_as_stored"]["variable"].value_counts()
print("Scores typed without their items, the ten most frequent:")
print(typed_alone.head(10))
print()

mismatch = flags[flags["flag"] == "score_mismatch"]["variable"].value_counts()
print("Scores whose stored value was wrong, the ten most frequent:")
print(mismatch.head(10))

**Several stored scores of this database are computed with a wrong rule**, and the tables hold the corrected ones: the totals A and C of the ADI-R, the scores of the SRS filled by a relative, two sub-scores of the ADOS module 4, three components of the PSQI and its total, two sub-scores of the CTQ, the state total of the STAIC, the factors of the BVAQ before 2025, the MARS total and the proprioceptive sub-score of the sensory questionnaire. The notes of each instrument explain (`notes("adi_r")`, `notes("ados_module4")`, `notes("psqi")`).

To see what was typed in a flagged cell, look at the columns `raw` and `value` of `flags`. The cell below only counts.

In [ ]:
one = flags[flags["variable"] == "psqi_total"]
print(len(one), "visits whose stored PSQI total differs from the recomputed one")
one["detail"].str.replace(r"[0-9.]+", "n", regex=True).value_counts().head(3)

## 7. The traps of this cohort

Read this section once. Each trap has cost someone a wrong result.

**Trap 1. A child form and an adult form are two instruments.** Anxiety is measured with the STAIC in children and the STAI in adults; depression with the CDI and the BDI-II; executive function with the BRIEF and the BRIEF-A; attention with the child form and the adult form of the ADHD-RS. They are different columns with different ranges. Never add them up or put them on one axis without saying so.

In [ ]:
pairs = {"cdi_total": "CDI (children)", "bdi_total": "BDI-II (adults)", "staic_trait": "STAIC (children)", "staya_total": "STAI (adults)",
         "briefp_gec": "BRIEF parent (children)", "briefa_gec": "BRIEF-A (adults)",
         "adhdrsc_total": "ADHD-RS child form (children)", "adhdrsa_total": "ADHD-RS adult form (adults)"}

inclusion.groupby("age_group")[list(pairs)].count().rename(columns=pairs).T

**Trap 2. Two generations of forms: "not asked" is not "missing".** Until about 2017 the centres used an old set of questionnaires (AQ, Empathy Quotient, RBS-R, Hamilton scales...). From 2019 to 2021 a new set replaced it (RAADS-R, BDI-II, GSQ...). An old scale is empty after the change because it was no longer asked. Always look at the years before you conclude that data are missing.

In [ ]:
by_year = screening.groupby(screening["visit_date"].dt.year)[["aq_total", "raads_total"]].agg(lambda column: column.notna().mean() * 100)

by_year.plot(figsize=(9, 3), marker="o", title="Screening visits with the AQ (old set) and the RAADS-R (new set), %")
plt.show()

**Trap 3. The ADOS has four modules, and each has its own columns.** The module depends on the age and the language of the patient. `ados3_total` and `ados4_total` are not on the same scale and do not have the same cut-offs. `ados_module` says which module was given.

In [ ]:
print(inclusion["ados_module"].value_counts())

pd.crosstab(inclusion["ados_module"], inclusion["age_group"])

**Trap 4. The codes 7, 8 and 9 of the ADI-R and ADOS items are not levels.** The items are delivered as recorded: 0 to 3, then 7 (an abnormality of another kind), 8 (not applicable) and 9 (not known). Never average the items. Use the algorithm scores, which count them the right way.

In [ ]:
inclusion["adi50_past"].value_counts().sort_index()

**Trap 5. The centres do not give the same instruments.** Before you compare centres on a score, check that they all fill it.

In [ ]:
share_by_centre = inclusion.groupby("siteid")[["adi_social_total", "ados4_total", "vineland_composite", "bdi_total", "cgi_severity"]].agg(lambda column: round(column.notna().mean() * 100))
share_by_centre

**Trap 6. The treatments are the list of the visit, and a missing list is not "no treatment".** In this cohort a treatment list belongs to a visit (in the other cohorts it is the history of the patient). And many visits declare a current treatment without any list.

In [ ]:
treatments = result.treatments
medications = treatments[treatments["list"] == "medications"]

with_list = set(medications.loc[medications["visit"] == "V0", "usubjid"])
declared = inclusion[inclusion["current_treatment"] == 1]

print("Inclusion visits that declare a current treatment:", len(declared))
print("  of which a list is recorded:", declared["usubjid"].isin(with_list).sum())

## 8. The few patients who are followed

About one patient in ten has a follow-up visit, almost all in three centres. The cell counts, for each visit, the patients seen.

In [ ]:
follow_up = data[~data["visit"].isin(["screening", "V0"])]

print(follow_up["usubjid"].nunique(), "patients have at least one follow-up visit")
print(follow_up["visit"].value_counts())

What can be followed? The instruments that are repeated: the ADOS module 3, the BRIEF of children, the ADHD-RS, the Short Sensory Profile, the CGI. The cell below draws the mean CGI severity by visit, for the patients who have a one-year visit. `groupby` makes one group per visit.

In [ ]:
seen_at_one_year = data.loc[data["visit"] == "V1_an", "usubjid"]
followed = data[data["usubjid"].isin(seen_at_one_year) & data["visit"].isin(["V0", "V1_an", "V2_ans"])]

print(followed.groupby("visit_month")["cgi_severity"].count())

followed.groupby("visit_month")["cgi_severity"].mean().plot(marker="o", figsize=(6, 3), title="Mean CGI severity, patients with a one-year visit")
plt.xlabel("Months since inclusion")
plt.show()

## 9. Use case 1: the diagnostic profile

**Question:** what do the two reference instruments, the ADI-R (interview of a parent) and the ADOS (observation of the patient), say, and how do they agree with the diagnosis?

**The ADI-R.** Three domain totals with a cut-off each: `adi_social_total` (A, cut-off 10), `adi_communication_verbal_total` (B, cut-off 8) and `adi_repetitive_total` (C, cut-off 3). The totals A and C in the tables are corrected: the stored ones counted too many items.

In [ ]:
domains = {"adi_social_total": 10, "adi_communication_verbal_total": 8, "adi_repetitive_total": 3}

figure, axes = plt.subplots(1, 3, figsize=(13, 3))
for axis, (name, cut_off) in zip(axes, domains.items()):
    inclusion[name].plot.hist(bins=20, ax=axis, title=name)
    axis.axvline(cut_off, color="black", linestyle="--")
plt.show()

A patient is above the three cut-offs when each total reaches its threshold. The cell builds that column and crosses it with the diagnosis. `&` means "and"; the comparison is only made where the three totals exist.

In [ ]:
has_adi = inclusion[list(domains)].notna().all(axis=1)
adi = inclusion[has_adi].copy()
adi["above_three_cut_offs"] = (adi["adi_social_total"] >= 10) & (adi["adi_communication_verbal_total"] >= 8) & (adi["adi_repetitive_total"] >= 3)

print(len(adi), "inclusion visits with the three ADI-R totals")
pd.crosstab(adi["above_three_cut_offs"], adi["diagnosis"], normalize="columns").round(2)

**The ADOS.** Each module has its classification: `autism`, `autism_spectrum` or `non_spectrum`. The module 3 is given to children and adolescents who speak fluently, the module 4 to adolescents and adults.

In [ ]:
print("Module 3:")
print(pd.crosstab(inclusion["ados3_classification"], inclusion["diagnosis"]))
print()
print("Module 4:")
print(pd.crosstab(inclusion["ados4_classification"], inclusion["diagnosis"]))

The two totals, drawn apart because they are not on the same scale. The dashed line is the cut-off of the autism spectrum (7 for both modules).

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 3))
for axis, name in zip(axes, ["ados3_total", "ados4_total"]):
    for label, group in inclusion.groupby("diagnosis"):
        group[name].plot.hist(bins=range(0, 28), alpha=0.5, ax=axis, label=label)
    axis.axvline(7, color="black", linestyle="--")
    axis.set_title(name)
    axis.legend()
plt.show()

**Do the interview and the observation agree?** `adi_positive` and `ados_positive` are the conclusions recorded by the clinician ("yes" or "no").

In [ ]:
pd.crosstab(inclusion["adi_positive"], inclusion["ados_positive"], margins=True)

**The age at the assessment.** The diagnosis comes late for many adults: the age at inclusion is the age at which the expert centre saw the patient.

In [ ]:
print(patients.groupby(["diagnosis", "sex"])["age_at_inclusion"].median())

patients.boxplot(column="age_at_inclusion", by="diagnosis", figsize=(6, 3))
plt.suptitle("")
plt.title("Age at inclusion, by diagnosis")
plt.show()

## 10. Use case 2: children and adults

**Question:** how do children and adults differ in what is measured and in what is found?

**Intelligence.** Children take the WISC (the fourth edition until 2016, the fifth from 2017), adults the WAIS-IV. Each edition has its own columns. The full-scale IQ is `wisc4_qit`, `wisc5_qit` or `wais4_qit`. They are standard scores (mean 100 in the general population) and are delivered as stored.

In [ ]:
print(inclusion[["wisc4_qit", "wisc5_qit", "wais4_qit"]].describe().loc[["count", "25%", "50%", "75%"]].round(0))

figure, axis = plt.subplots(figsize=(9, 3))
for name in ["wisc5_qit", "wais4_qit"]:
    inclusion[name].plot.hist(bins=range(40, 161, 5), alpha=0.5, ax=axis, label=name)
plt.axvline(100, color="black", linestyle="--")
plt.title("Full-scale IQ: WISC-V (children) and WAIS-IV (adults)")
plt.legend()
plt.show()

**Adaptive behaviour.** The Vineland-II measures what the patient does in daily life: `vineland_composite` is a standard score too. In autism it is often far below the IQ. The cell draws one against the other for the adults.

In [ ]:
both = inclusion[inclusion["wais4_qit"].notna() & inclusion["vineland_composite"].notna()]
print(len(both), "adults with a WAIS-IV IQ and a Vineland composite")

both.plot.scatter(x="wais4_qit", y="vineland_composite", alpha=0.3, figsize=(5, 4), title="Adults: IQ and adaptive behaviour")
plt.show()

print("Median IQ:", both["wais4_qit"].median(), "| median Vineland composite:", both["vineland_composite"].median())

**School.** `assisted_schooling` says whether the patient had a support at school (1 yes, 0 no; the column `scolass` of the extraction). The section has descriptive names since the review of the small tabs (`find("school")` lists them).

In [ ]:
school = inclusion[inclusion["assisted_schooling"].notna()]
print((school.groupby("age_group")["assisted_schooling"].mean() * 100).round(0))

find("Schooling").head(12)

**Which questionnaires for whom.** The table shows, for a few instruments, the share of children and of adults who have them. A zero is a form that is not given at that age, not missing data.

In [ ]:
some = {"adhdrsc_total": "ADHD-RS, child form", "adhdrsa_total": "ADHD-RS, adult form", "asrs_total": "ASRS", "ssp_total": "Short Sensory Profile", "gsq_total": "GSQ", "aqc_total": "Alexithymia, children", "bvaq01": "BVAQ",
        "cdi_total": "CDI", "bdi_total": "BDI-II", "ctq_total": "CTQ", "eq5d_vas": "EQ-5D"}

(inclusion.groupby("age_group")[list(some)].agg(lambda column: round(column.notna().mean() * 100)).rename(columns=some).T)

## 11. Use case 3: comorbidities and treatments

**Question:** which psychiatric disorders come with autism in this cohort, and what are the patients treated with?

**The diagnoses of the medical evaluation.** The clinician records, for each disorder, 1 (present, now or in the past) or 0. `depressive_episode_v0`, `anxiety_disorder_v0`, `adhd_v0` and `insomnia_v0` are the main ones (the ending `_v0` is the list at inclusion; the follow-up list is a schema of its own, `_fu`).

In [ ]:
disorders = ["depressive_episode_v0", "anxiety_disorder_v0", "adhd_v0", "insomnia_v0"]

asd = inclusion[inclusion["diagnosis"] == "asd_confirmed"]
shares = (asd.groupby("age_group")[disorders].mean() * 100).round(0)
print(asd.groupby("age_group")[disorders].count())

shares.T.plot.bar(figsize=(8, 3), title="Patients with a confirmed ASD: share with each disorder (%)")
plt.show()

**The scales.** Depression is measured by the BDI-II in adults (0 to 63; 20 and more is at least moderate) and by the CDI in children (0 to 54). Compare each with its own disorder column: the scale is the patient's answer today, the diagnosis is the clinician's judgement over the life.

In [ ]:
print(inclusion.groupby("depressive_episode_v0")["bdi_total"].describe()[["count", "25%", "50%", "75%"]])
print()
print(inclusion.groupby("depressive_episode_v0")["cdi_total"].describe()[["count", "25%", "50%", "75%"]])

**Attention.** The ADHD-RS (0 to 54) is two forms in the same columns, told apart by the age at the visit: the child form, in which a parent rates the behaviours of the child (`adhdrsc_total`), and the adult form, in which the adult recalls them as a child (`adhdrsa_total`). The clinician's diagnosis is `adhd_v0`.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
for axis, column, title in zip(axes, ["adhdrsc_total", "adhdrsa_total"], ["Child form (a parent rates)", "Adult form (recalls childhood)"]):
    inclusion.boxplot(column=column, by="adhd_v0", ax=axis)
    axis.set_title(title)
plt.suptitle("ADHD-RS total, by the clinician's diagnosis of ADHD (0 no, 1 yes)")
plt.show()

**The treatments.** `result.treatments` holds eleven lists; the column `list` says which. We keep the medications recorded at the inclusion visit. Each row is one treatment, with its `molecule` and its `treatment_group` (looked up from the name that was typed).

In [ ]:
print(treatments["list"].value_counts())

at_inclusion = medications[medications["visit"] == "V0"]
print()
print(at_inclusion["usubjid"].nunique(), "patients have a treatment list at inclusion")
at_inclusion["molecule"].value_counts().head(12)

One row per patient and group, then the share of the patients **who have a list** on each group, for children and adults. Remember trap 6: the patients without a list are not counted here, and they are not all untreated.

In [ ]:
groups = at_inclusion.dropna(subset=["treatment_group"]).drop_duplicates(["usubjid", "treatment_group"])
groups = groups.merge(patients[["usubjid", "age_group"]], on="usubjid")

listed = at_inclusion.drop_duplicates("usubjid").merge(patients[["usubjid", "age_group"]], on="usubjid")["age_group"].value_counts()
table = pd.crosstab(groups["treatment_group"], groups["age_group"])

(table / listed * 100).round(0).sort_values("adult", ascending=False)

**Psychiatric history.** `psychiatric_contact` (an earlier contact with psychiatry) and `psychiatric_hospitalisation`; the hospitalisations themselves are a list of `treatments`, one row per stay.

In [ ]:
print((inclusion.groupby("age_group")[["psychiatric_contact", "psychiatric_hospitalisation"]].mean() * 100).round(0))

stays = treatments[treatments["list"] == "hospitalisations"]
print()
print(len(stays), "hospitalisations of", stays["usubjid"].nunique(), "patients; median length in days:", stays["hospitalisation_days"].median())

## 12. Use case 4: self-reported traits

**Question:** what do the patients, or their relatives, report about autistic traits, the senses, executive function and emotions?

**Autistic traits.** Adults fill the RAADS-R at the screening visit (`raads_total`, 0 to 240; 65 and more is the published threshold). A relative fills the SRS (0 to 195): the child form for a patient under 18 (`srsc_total`), the adult form otherwise (`srsr_total`); adults also fill it themselves (`srsa_total`). The two forms filled by a relative have different norms: do not put them in one column to compare a child with an adult. The old forms used the AQ (`aq_total`, 0 to 50).

In [ ]:
print(screening[["raads_total", "srsa_total", "srsr_total", "srsc_total", "aq_total"]].describe().loc[["count", "25%", "50%", "75%"]].round(0))

figure, axis = plt.subplots(figsize=(9, 3))
for label, group in screening.groupby("diagnosis"):
    group["raads_total"].plot.hist(bins=range(0, 241, 10), alpha=0.5, ax=axis, label=label)
plt.axvline(65, color="black", linestyle="--")
plt.title("RAADS-R at screening, by final diagnosis")
plt.legend()
plt.show()

**Self-report and relative.** For adults who have both forms of the SRS at screening: do the patient and the relative agree? The two totals are on the same scale.

In [ ]:
two_forms = screening[screening["srsa_total"].notna() & screening["srsr_total"].notna()]
print(len(two_forms), "adults with the SRS filled by themselves and by a relative")

two_forms.plot.scatter(x="srsa_total", y="srsr_total", alpha=0.3, figsize=(5, 4), title="SRS: self-report and relative")
plt.show()

print("Correlation:", round(two_forms["srsa_total"].corr(two_forms["srsr_total"]), 2))

**The senses.** Adults fill the Glasgow Sensory Questionnaire (`gsq_total`, higher is more atypical); a parent fills the Short Sensory Profile for children (`ssp_total`, **lower** is more atypical). Two instruments, two directions.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 3))
inclusion["gsq_total"].plot.hist(bins=25, ax=axes[0], title="GSQ, adults (higher is more atypical)")
inclusion["ssp_total"].plot.hist(bins=25, ax=axes[1], title="Short Sensory Profile, children (lower is more atypical)")
plt.show()

**Executive function.** The BRIEF has two forms, and `brief_form` says which one a visit holds. The composite is `briefp_gec` for the form filled by a parent and `briefa_gec` for the adult form. They are raw sums with different numbers of items: do not compare their values.

In [ ]:
print(inclusion["brief_form"].value_counts())
print()
print(inclusion[["briefp_gec", "briefa_gec"]].describe().loc[["count", "25%", "50%", "75%"]].round(0))

**Emotions.** Children fill the Alexithymia Questionnaire for Children (`aqc_total`, 0 to 40). Do traits and alexithymia go together? The child form of the SRS is filled by a parent at screening and the alexithymia questionnaire at inclusion: we bring the screening score next to the inclusion row with `merge`.

In [ ]:
srs_at_screening = screening[["usubjid", "srsc_total"]].dropna()
children = inclusion[["usubjid", "aqc_total", "cdi_total"]].merge(srs_at_screening, on="usubjid")

print(children[["aqc_total", "cdi_total", "srsc_total"]].count())
children[["aqc_total", "cdi_total", "srsc_total"]].corr().round(2)

## 13. Build your own table and save it

Choose the columns you need, add the patient columns, and save. The file goes to `data/exports/`, which stays on this computer.

In [ ]:
wanted = ["usubjid", "visit", "visit_date", "siteid", "age", "adi_social_total", "adi_communication_verbal_total", "adi_repetitive_total", "ados_module", "ados3_total", "ados4_total",
          "cgi_severity", "gaf", "wais4_qit", "wisc5_qit", "vineland_composite"]

mine = data.loc[data["visit"] == "V0", wanted].merge(patients[["usubjid", "sex", "diagnosis", "age_at_inclusion"]], on="usubjid")

Path("data/exports").mkdir(parents=True, exist_ok=True)
mine.to_csv("data/exports/asp_diagnostic_profile.csv", index=False)

print(len(mine), "patients and", mine.shape[1], "columns saved")

## 14. The four cohorts in one table

The bipolar (`bp`), schizophrenia (`sz`), resistant depression (`dr`) and autism (`asp`) cohorts are loaded by the same package. The instruments that several cohorts have keep the same name and the same columns, so the tables can be put under one another with `pd.concat`.

**Step 1: which instruments have the same name in the four cohorts?**

In [ ]:
from facedata import bp, sz, dr

names = {}
for label, module in [("bp", bp), ("sz", sz), ("dr", dr), ("asp", asp)]:
    names[label] = {file.stem for file in Path(module.SCHEMAS).glob("*/*.yaml")}

in_all_four = sorted(names["bp"] & names["sz"] & names["dr"] & names["asp"])
print(len(in_all_four), "instruments have the same name in the four cohorts:")
print(in_all_four)

**Step 2: load the same instruments from the four files.** We take the identity, the CGI and the GAF, the childhood trauma questionnaire and the EQ-5D. `instruments=` loads only these: it takes a minute or two, not ten.

In [ ]:
wanted = ["identite", "cgi_egf", "ctq", "eq5d"]

bipolar = bp.load("data/bp/BPALL_20261113_2.tsv", instruments=wanted)
schizophrenia = sz.load("data/sz/SZALL_20261123.tsv", instruments=wanted)
depression = dr.load("data/dr/DRALL_20261030.tsv", instruments=wanted)
autism = asp.load(EXTRACTION, instruments=wanted)

**Step 3: stack.** The first column of every table is `cohort`. **A `usubjid` is unique inside a cohort only**: the same number can be another patient in another cohort. The key is `cohort` with `usubjid`.

In [ ]:
four = pd.concat([bipolar.data, schizophrenia.data, depression.data, autism.data], ignore_index=True)
four_patients = pd.concat([bipolar.patients, schizophrenia.patients, depression.patients, autism.patients], ignore_index=True)

print(four.groupby("cohort")["usubjid"].nunique())
print()
print("Same number in several cohorts:", four_patients["usubjid"].duplicated().sum(), "| same cohort and number:", four_patients.duplicated(["cohort", "usubjid"]).sum())

The inclusion visits of the four cohorts, with the sex and the age of the patients. The join uses the two key columns.

In [ ]:
first_four = four[four["visit"] == "V0"].merge(four_patients[["cohort", "usubjid", "sex", "age_at_inclusion"]], on=["cohort", "usubjid"])

print(first_four.groupby("cohort")["age_at_inclusion"].describe()[["count", "25%", "50%", "75%"]].round(0))
print()
(pd.crosstab(first_four["cohort"], first_four["sex"], normalize="index") * 100).round(0)

The clinician's global ratings, by cohort. The CGI severity goes from 1 (normal) to 7; the GAF from 1 to 100 (higher is better).

In [ ]:
print(first_four.groupby("cohort")[["cgi_severity", "gaf"]].count())

first_four.boxplot(column="gaf", by="cohort", figsize=(7, 3))
plt.suptitle("")
plt.title("GAF at inclusion, by cohort")
plt.show()

Childhood trauma, for the adults who filled the questionnaire. `density=True` makes the histograms comparable although the cohorts do not have the same size.

In [ ]:
for name, group in first_four.groupby("cohort"):
    if group["ctq_total"].notna().sum() >= 30:
        group["ctq_total"].plot.hist(bins=30, alpha=0.5, density=True, label=name)

plt.title("Childhood trauma questionnaire, total score at inclusion")
plt.legend()
plt.show()

**Three things to check before you compare the cohorts.**

1. **A column that only some cohorts have is empty for the others**, and in the autism cohort an adult questionnaire is empty for every child. Count the values per cohort before you use a column.
2. **The same name does not always mean exactly the same form.** The file `facedata/asp/differences.yaml` lists, for each shared instrument, what differs in the autism cohort. The cell below prints it for the instruments we loaded.
3. **The cohorts were not recruited the same way**: the autism cohort has children, a group of patients not diagnosed, and almost no follow-up. A difference between cohorts is a description, not an effect of the diagnosis.

In [ ]:
print(four.groupby("cohort")[["gaf", "cgi_severity", "ctq_total", "eq5d_vas"]].count())
print()

differences = yaml.load(open("facedata/asp/differences.yaml", encoding="utf-8"), Loader=yaml.BaseLoader)
for name in wanted:
    print(name, ":", differences.get(name, "the same as in the cohort it is shared with"))

**Save the stacked table**, as in section 13.

In [ ]:
first_four.to_csv("data/exports/four_cohorts_inclusion.csv", index=False)

print(len(first_four), "patients of", first_four["cohort"].nunique(), "cohorts and", first_four.shape[1], "columns saved")

## Where to go next

- The notes of each instrument (section 5): `notes("adi_r")`, `notes("ados_module4")`, `notes("srs_relative_child")`, `notes("psqi")`, `notes("medications")`.
- The quality report: `asp.report(result, "reports/asp")` writes one page per instrument, with the distribution of each variable by year and by centre.
- `README.md` for the package, and `docs/asp/PROGRESS.md` for every decision taken about the data and every open question.
- `bipolar-tutorial.ipynb`, `schizophrenia-tutorial.ipynb` and `depression-tutorial.ipynb`, the same tour for the three other cohorts.
- The five moves used in this notebook cover most needs: keep rows with `table[table["column"] == value]`, count with `value_counts()`, summarise with `groupby(...).mean()`, join with `merge`, and draw with `.plot`.

**Before you close: clear the results** (menu *Edit*, *Clear Outputs of All Cells*) if this notebook will be shared.